In [0]:
# Databricks notebook source
# DBFS / Catalog Configuration
# CATALOG = "workspace"
# SCHEMA = "default"

from pyspark.sql import functions as F
from pyspark.sql.window import Window

# -------------------------------------------------------------------------
# 1. CARGA DAS TABELAS BRUTAS
# -------------------------------------------------------------------------
df_pipeline = spark.table("workspace.default.sales_pipeline")
df_accounts = spark.table("workspace.default.accounts")
df_products = spark.table("workspace.default.products")
df_teams = spark.table("workspace.default.sales_teams")

# -------------------------------------------------------------------------
# 2. SANEAMENTO E DENORMALIZAÇÃO (FILTRO PARA DEALS ABERTOS)
# -------------------------------------------------------------------------
# Consideramos 'Prospecting' e 'Engaging' como oportunidades ativas
df_active_deals = df_pipeline.filter(F.col("deal_stage").isin("Prospecting", "Engaging"))

# Data de referência para cálculo de idade (Ageing)
# Usa a maior data do pipeline se houver registro futuro, ou a data atual (current_date)
max_engage_date = df_pipeline.select(F.max("engage_date")).collect()[0][0]

df_joined = df_active_deals.alias("sp") \
    .join(df_accounts.alias("acc"), F.col("sp.account") == F.col("acc.account"), "left") \
    .join(df_products.alias("prod"), F.col("sp.product") == F.col("prod.product"), "left") \
    .join(df_teams.alias("team"), F.col("sp.sales_agent") == F.col("team.sales_agent"), "left") \
    .select(
        F.col("sp.opportunity_id"),
        F.col("sp.sales_agent"),
        F.col("team.manager"),
        F.col("team.regional_office"),
        F.col("sp.account"),
        F.col("acc.sector"),
        F.col("acc.revenue"),
        F.col("acc.employees"),
        F.col("acc.office_location").alias("account_location"),
        F.col("sp.product"),
        F.col("prod.series"),
        F.col("prod.sales_price"),
        F.col("sp.deal_stage"),
        F.col("sp.engage_date"),
        # Calcula a idade do deal em dias considerando a data de engajamento
        F.datediff(F.lit(max_engage_date), F.col("sp.engage_date")).alias("days_in_pipeline")
    )

# -------------------------------------------------------------------------
# 3. APLICAÇÃO DAS REGRAS DE LEAD SCORING (PILAR A PILAR)
# -------------------------------------------------------------------------

# Pilar 1: Estágio (Max: 25 pts)
score_stage = F.when(F.col("deal_stage") == "Engaging", 25) \
               .when(F.col("deal_stage") == "Prospecting", 10) \
               .otherwise(0)

# Pilar 2: Ageing / Timing (Max: 30 pts, Penalidade: -15 pts)
score_timing = F.when(F.col("days_in_pipeline") <= 30, 30) \
                .when((F.col("days_in_pipeline") > 30) & (F.col("days_in_pipeline") <= 60), 20) \
                .when((F.col("days_in_pipeline") > 60) & (F.col("days_in_pipeline") <= 90), 10) \
                .otherwise(-15)

# Pilar 3.1: Setor (Max: 15 pts)
top_sectors = ["marketing", "entertainment", "software"]
mid_sectors = ["technology", "services", "retail"]

score_sector = F.when(F.lower(F.col("sector")).isin(top_sectors), 15) \
                .when(F.lower(F.col("sector")).isin(mid_sectors), 10) \
                .otherwise(5)

# Pilar 3.2: Porte da Empresa / Receita (Max: 10 pts)
score_revenue = F.when(F.col("revenue") > 1000, 10).otherwise(5)

# Pilar 4: Produto / Ticket (Max: 20 pts)
score_product = F.when(F.col("sales_price") >= 3000, 20).otherwise(10)

# Sumatório do Score Final (Limitado entre 0 e 100)
raw_score = score_stage + score_timing + score_sector + score_revenue + score_product
final_score = F.when(raw_score > 100, 100).when(raw_score < 0, 0).otherwise(raw_score)

# -------------------------------------------------------------------------
# 4. EXPLICABILIDADE E TIERS
# -------------------------------------------------------------------------

# Tier de Prioridade
priority_tier = F.when(final_score >= 80, "High Priority") \
                 .when(final_score >= 50, "Medium Priority") \
                 .otherwise("Low Priority")

# Tags Explicativas
pos_tag = F.when(F.lower(F.col("sector")).isin(top_sectors), "Setor com alta taxa histórica de conversão") \
           .when(F.col("sales_price") >= 3000, "Produto de alto ticket de venda") \
           .otherwise("Cliente do segmento estratégico")

neg_tag = F.when(F.col("days_in_pipeline") > 90, "Deal parado no pipeline há mais de 90 dias (risco de estagnação)") \
           .when(F.col("deal_stage") == "Prospecting", "Estágio inicial (requer avançar qualificação)") \
           .otherwise("Sem alertas críticos de estagnação")

action_tag = F.when(final_score >= 80, "Entrar em contato nas próximas 24 horas") \
              .when(final_score >= 50, "Agendar acompanhamento para os próximos 3 dias") \
              .otherwise("Avaliar arquivamento ou reengajamento pontual")

# -------------------------------------------------------------------------
# 5. MONTAGEM DA TABELA FINAL
# -------------------------------------------------------------------------
df_lead_scoring = df_joined.withColumn("score_stage", score_stage) \
                           .withColumn("score_timing", score_timing) \
                           .withColumn("score_sector", score_sector) \
                           .withColumn("score_revenue", score_revenue) \
                           .withColumn("score_product", score_product) \
                           .withColumn("lead_score", final_score) \
                           .withColumn("priority_tier", priority_tier) \
                           .withColumn("reason_positive", pos_tag) \
                           .withColumn("reason_negative", neg_tag) \
                           .withColumn("recommended_action", action_tag) \
                           .withColumn("calculated_at", F.current_timestamp())

# -------------------------------------------------------------------------
# 6. GRAVAÇÃO DA TABELA CONSOLIDADA NO DELTA LAKE
# -------------------------------------------------------------------------
output_table = "workspace.default.lead_scoring_pipeline"

df_lead_scoring.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(output_table)

print(f"Tabela consolidada gravada com sucesso em: {output_table}")

# Exibir prévia ordenada pelos maiores scores
display(df_lead_scoring.orderBy(F.col("lead_score").desc()))

Tabela consolidada gravada com sucesso em: workspace.default.lead_scoring_pipeline


opportunity_id,sales_agent,manager,regional_office,account,sector,revenue,employees,account_location,product,series,sales_price,deal_stage,engage_date,days_in_pipeline,score_stage,score_timing,score_sector,score_revenue,score_product,lead_score,priority_tier,reason_positive,reason_negative,recommended_action,calculated_at
K9KGXL2E,Markita Hansen,Celia Rouche,West,Lexiqvolax,medical,1618.89,3889,United States,MG Advanced,MG,3393,Engaging,2017-12-13,14,25,30,5,10,20,90,High Priority,Produto de alto ticket de venda,Sem alertas críticos de estagnação,Entrar em contato nas próximas 24 horas,2026-10-16T23:28:35.346Z
L8XNQONA,Zane Levy,Summer Sewald,West,Inity,marketing,2403.58,8801,United States,GTX Basic,GTX,550,Engaging,2017-11-27,30,25,30,15,10,10,90,High Priority,Setor com alta taxa histórica de conversão,Sem alertas críticos de estagnação,Entrar em contato nas próximas 24 horas,2026-10-16T23:28:35.346Z
RDHTQLNI,Cassey Cress,Rocco Neubert,East,null,null,null,null,null,GTX Plus Pro,GTX,5482,Engaging,2017-12-21,6,25,30,5,5,20,85,High Priority,Produto de alto ticket de venda,Sem alertas críticos de estagnação,Entrar em contato nas próximas 24 horas,2026-10-16T23:28:35.346Z
FZWU2I30,Zane Levy,Summer Sewald,West,The New York Inquirer,medical,439.21,792,United States,GTX Plus Pro,GTX,5482,Engaging,2017-12-10,17,25,30,5,5,20,85,High Priority,Produto de alto ticket de venda,Sem alertas críticos de estagnação,Entrar em contato nas próximas 24 horas,2026-10-16T23:28:35.346Z
GQDS81G1,Corliss Cosme,Cara Losch,East,null,null,null,null,null,GTX Plus Pro,GTX,5482,Engaging,2017-11-30,27,25,30,5,5,20,85,High Priority,Produto de alto ticket de venda,Sem alertas críticos de estagnação,Entrar em contato nas próximas 24 horas,2026-10-16T23:28:35.346Z
RTWE19SU,Kary Hendrixson,Summer Sewald,West,Conecom,technolgy,1520.66,1806,United States,GTXPro,null,null,Engaging,2017-12-17,10,25,30,5,10,10,80,High Priority,Cliente do segmento estratégico,Sem alertas críticos de estagnação,Entrar em contato nas próximas 24 horas,2026-10-16T23:28:35.346Z
7XRWWM0X,Kami Bicknell,Summer Sewald,West,Scotfind,software,6354.87,16780,United States,MG Special,MG,55,Engaging,2017-11-15,42,25,20,15,10,10,80,High Priority,Setor com alta taxa histórica de conversão,Sem alertas críticos de estagnação,Entrar em contato nas próximas 24 horas,2026-10-16T23:28:35.346Z
J1G2SZKD,Zane Levy,Summer Sewald,West,Goodsilron,marketing,2952.73,5107,United States,GTX Basic,GTX,550,Engaging,2017-11-22,35,25,20,15,10,10,80,High Priority,Setor com alta taxa histórica de conversão,Sem alertas críticos de estagnação,Entrar em contato nas próximas 24 horas,2026-10-16T23:28:35.346Z
HB740BLB,Hayden Neloms,Celia Rouche,West,Silis,medical,2818.38,6290,United States,MG Special,MG,55,Engaging,2017-12-20,7,25,30,5,10,10,80,High Priority,Cliente do segmento estratégico,Sem alertas críticos de estagnação,Entrar em contato nas próximas 24 horas,2026-10-16T23:28:35.346Z
PM2BF2KB,Cassey Cress,Rocco Neubert,East,Xx-holding,finance,7537.24,20293,United States,GTX Plus Basic,GTX,1096,Engaging,2017-12-13,14,25,30,5,10,10,80,High Priority,Cliente do segmento estratégico,Sem alertas críticos de estagnação,Entrar em contato nas próximas 24 horas,2026-10-16T23:28:35.346Z
